# Aspect-Based Sentiment Analysis & Opinion Mining (Advanced)
## CDS6344 Social Media Computing - Restaurant Review Analysiss

### 7 Aspect Categories:
1. **Food** - Dishes, taste, flavor, cuisine
2. **Service** - Staff, waiter, hospitality
3. **Waiting Time** - Wait, queue, delay, speed
4. **Ambience** - Atmosphere, decor, lighting, music
5. **Cleanliness** - Clean, hygiene, toilet
6. **Price** - Cost, value, bill
7. **Location** - Parking, access, convenient

### Features:
- Proper dependency parsing for (Target, Opinion_Expression, Sentiment) triplets
- Negation handling ("not good", "not fresh")
- Platform comparison (Google Reviews vs TripAdvisor)
- Comprehensive visualizations & summaries

## Part 1: Setup and Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from collections import defaultdict, Counter
import re
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import opinion_lexicon
import os 
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('vader_lexicon', quiet=True)
nltk.download('opinion_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

try:
    nlp = spacy.load('en_core_web_sm')
except:
    print('Downloading spaCy model...')
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

sia = SentimentIntensityAnalyzer()
print('✓ All libraries loaded')

✓ All libraries loaded


In [5]:
# Read raw data
master_dir = os.path.dirname(os.getcwd())

processed_reviews = os.path.join(master_dir, "data", "processed_data", "feature_engineered_reviews.csv")

df = pd.read_csv(processed_reviews)

df.head()

,Review_ID,Review,Rating,Lemmatized_Tokens,Review_lemmatized,Platform,Restaurant,Sentiment_Label
0,R000001,food is taste good and the environment also no...,3.0,"['food', 'taste', 'good', 'environment', 'also...",food taste good environment also not bad . how...,Google,Tandoor Grill,Neutral
1,R000002,our family celebrated birthday on last weekend...,1.0,"['family', 'celebrate', 'birthday', 'last', 'w...",family celebrate birthday last weekend think 5...,Tripadvisor,Latest Recipe,Negative
2,R000003,pretty nice nonya food but very commercial alr...,3.0,"['pretty', 'nice', 'nonya', 'food', 'but', 've...",pretty nice nonya food but very commercial alr...,Google,Madam Kwan's Suria KLCC,Neutral
3,R000004,"food was not on point, the seafood was frozen,...",2.0,"['food', 'not', 'point', ',', 'seafood', 'froz...","food not point , seafood frozen , seasoning , ...",Google,Mercat Barcelona Gastrobar (1MontKiara),Negative
4,R000005,personaly for me is a big no....i saw flies la...,1.0,"['personaly', 'big', 'no', '....', 'saw', 'fly...",personaly big no .... saw fly lay egg fry chic...,Google,Restoran DEEN | Nasi Kandar,Negative


## Part 2: Opinion Triplet Extraction with Dependency Parsing & Negation

In [6]:
def extract_opinion_triplets_ultrafast(text):
    """
    ULTRA-FAST extraction (no sentiment scoring during extraction)
    Sentiment is scored later in bulk for speed.
    """
    triplets = []
    
    # Rule 1: Adjectival Modifier (amod)
    for token in text:
        if token.pos_ in ['NOUN', 'PROPN']:
            for child in token.children:
                if child.dep_ == 'amod' and child.pos_ == 'ADJ':
                    neg = ''.join([g.text for g in child.children if g.dep_ == 'neg'])
                    opinion = (neg + ' ' + child.text).strip() if neg else child.text
                    triplets.append({'Target': token.text, 'Opinion_Expression': opinion})
    
    # Rule 2: Adjectival Complement (acomp)
    for token in text:
        if token.pos_ == 'ADJ' and token.dep_ == 'acomp':
            for sibling in token.head.children:
                if sibling.dep_ == 'nsubj':
                    neg = ''.join([c.text for c in token.children if c.dep_ == 'neg'])
                    opinion = (neg + ' ' + token.text).strip() if neg else token.text
                    triplets.append({'Target': sibling.text, 'Opinion_Expression': opinion})
                    break
    
    return triplets

# ULTRA-FAST BATCH EXTRACTION (skip sentiment for now)
print('Ultra-fast extraction with batch processing...')

docs = list(nlp.pipe(df['Review'].apply(lambda x: x.lower()), batch_size=50, n_process=1))
df['Opinion_Triplets'] = [extract_opinion_triplets_ultrafast(doc) for doc in docs]

# Flatten
all_triplets = []
for idx, triplets in enumerate(df['Opinion_Triplets']):
    for triplet in triplets:
        triplet['Review_ID'] = idx
        all_triplets.append(triplet)

triplet_df = pd.DataFrame(all_triplets)
print(f'✓ Extracted {len(triplet_df)} triplets in seconds!')

# NOW score sentiment in bulk (much faster)
print('Scoring sentiment (batch mode)...')
triplet_df['Polarity_Score'] = triplet_df['Opinion_Expression'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)
triplet_df['Sentiment'] = triplet_df['Polarity_Score'].apply(
    lambda x: 'Positive' if x > 0.05 else ('Negative' if x < -0.05 else 'Neutral')
)

print(f'\n✓ COMPLETE!')
print(f'Total triplets: {len(triplet_df)}')
print(f'Sample:')
print(triplet_df.head(10))


Ultra-fast extraction with batch processing...
✓ Extracted 436547 triplets in seconds!
Scoring sentiment (batch mode)...

✓ COMPLETE!
Total triplets: 436547
Sample:
    Target Opinion_Expression  Review_ID  Polarity_Score Sentiment
0    event            certain          0          0.2732  Positive
1  service               slow          0          0.0000   Neutral
2  service               slow          0          0.0000   Neutral
3  weekend               last          1          0.0000   Neutral
4  company           previous          1          0.0000   Neutral
5  compare           not good          1         -0.3412  Negative
6    water              plain          1          0.0000   Neutral
7     tray              small          1          0.0000   Neutral
8   shrimp              small          1          0.0000   Neutral
9   plates               good          1          0.4404  Positive


In [7]:
# Verify negation handling
print('\nNegation Handling Examples:')
print('='*70)

negation_examples = triplet_df[triplet_df['Opinion_Expression'].str.contains('not', case=False)]
print(f'\nTriples with negation ("not"): {len(negation_examples)}')
if len(negation_examples) > 0:
    print('\nExamples:')
    for _, row in negation_examples.head(5).iterrows():
        print(f"  Target: '{row['Target']}' | Opinion: '{row['Opinion_Expression']}' | Score: {row['Polarity_Score']:.3f} | Sentiment: {row['Sentiment']}")
else:
    print('No negation examples found in dataset')


Negation Handling Examples:

Triples with negation ("not"): 3976

Examples:
  Target: 'compare' | Opinion: 'not good' | Score: -0.341 | Sentiment: Negative
  Target: 'price' | Opinion: 'not expensive' | Score: 0.000 | Sentiment: Neutral
  Target: 'choices' | Opinion: 'not many' | Score: 0.000 | Sentiment: Neutral
  Target: 'rm10)' | Opinion: 'not good' | Score: -0.341 | Sentiment: Negative
  Target: 'food' | Opinion: 'not bad' | Score: 0.431 | Sentiment: Positive


## Part 3: Aspect Categorization (7 Categories)

In [8]:
# Define aspect keywords - 7 categories
aspect_keywords = {
    'Food': [
        'food', 'dish', 'meal', 'taste', 'flavor', 'flavour', 'cuisine',
        'rice', 'curry', 'noodle', 'pasta', 'meat', 'chicken', 'fish',
        'beef', 'vegetable', 'sauce', 'spice', 'pizza', 'burger',
        'steak', 'salmon', 'prawn', 'dessert', 'cake', 'coffee', 'tea',
        'bread'
    ],

    'Service': [
        'service', 'staff', 'waiter', 'waitress', 'server',
        'hospitality', 'attention', 'attentive', 'helpful',
        'friendly', 'polite', 'rude'
    ],

    'Waiting Time': [
        'wait', 'waiting', 'queue', 'delay', 'slow', 'quick',
        'fast', 'time'
    ],

    'Ambience': [
        'ambience', 'atmosphere', 'ambiance', 'decor', 'decoration',
        'cozy', 'crowded', 'noisy', 'quiet', 'lighting', 'music',
        'interior'
    ],

    'Cleanliness': [
        'clean', 'cleanliness', 'dirty', 'hygiene', 'toilet',
        'washroom'
    ],

    'Price': [
        'price', 'cost', 'expensive', 'cheap', 'value', 'money',
        'bill', 'amount', 'charge', 'worth', 'afford', 'overpriced'
    ],

    'Location': [
        'location', 'address', 'parking', 'access', 'convenient',
        'near', 'far', 'distance', 'area'
    ]
}

print('Aspect Keywords Loaded:')
for category, keywords in aspect_keywords.items():
    print(f'  {category}: {len(keywords)} keywords')

Aspect Keywords Loaded:
  Food: 28 keywords
  Service: 12 keywords
  Waiting Time: 8 keywords
  Ambience: 12 keywords
  Cleanliness: 6 keywords
  Price: 12 keywords
  Location: 9 keywords


## Part 4: Platform Comparison (Google vs TripAdvisor)

## Part 5: Visualizations (4 PNG Files)